# Ankit_Task4_Movie_Recommenders
## Movie Recommender System — MovieLens 20M

**Methods implemented:**
1. Popularity-Based Recommender (Bayesian Average)
2. Content-Based Filtering — TF-IDF + TruncatedSVD → **Latent Matrix 1**
3. Collaborative Filtering — User-Movie Matrix + TruncatedSVD → **Latent Matrix 2**
4. Matrix Factorization — **Surprise SVD**
5. Hybrid Recommender (Content + Collab + Personalisation)

**Files used:** `movies.csv` · `ratings.csv` · `tags.csv` · `genome-tags.csv` · `links.csv`  
**Source:** https://grouplens.org/datasets/movielens/20m/


## Cell 1 — Install Dependencies
> Run this cell, then **Kernel → Restart**, then run all remaining cells.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# numpy < 2.0 required — scikit-surprise's Cython extensions break on numpy 2.x
pip("install", "--quiet", "numpy<2.0")
pip("install", "--quiet", "--no-cache-dir", "--force-reinstall", "scikit-surprise")
pip("install", "--quiet", "scikit-learn", "pandas", "scipy")

import numpy, sklearn, surprise
print(f"numpy    {numpy.__version__}")
print(f"sklearn  {sklearn.__version__}")
print(f"surprise {surprise.__version__}")
print()
print("✔ All packages ready.")
print(">>> NOW: Kernel → Restart Kernel, then Run All Cells <<<")

## Cell 2 — Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate
from surprise.model_selection import train_test_split as surprise_split
from surprise import accuracy

print("✔ All imports successful.")

## Cell 3 — Configuration
> Set `DATA_PATH` to the folder containing your CSV files.

In [ ]:
# ─── Point this to the folder containing all 5 CSV files ──────────────────
# Examples:
#   DATA_PATH = './'          # same folder as notebook
#   DATA_PATH = './ml-20m/'   # subfolder named ml-20m
DATA_PATH = './'

# ─── Collaborative filter thresholds ──────────────────────────────────────
# Lower values = faster but noisier. Increase for 20M dataset on GPU.
MIN_MOVIE_RATINGS = 50    # movies need at least this many ratings
MIN_USER_RATINGS  = 20    # users need at least this many ratings

# ─── Latent dimensions ─────────────────────────────────────────────────────
N_CONTENT_DIMS = 100   # SVD dims for content  (Latent Matrix 1)
N_COLLAB_DIMS  = 50    # SVD dims for collab   (Latent Matrix 2)

print(f"DATA_PATH        : {DATA_PATH}")
print(f"MIN_MOVIE_RATINGS: {MIN_MOVIE_RATINGS}")
print(f"MIN_USER_RATINGS : {MIN_USER_RATINGS}")
print(f"N_CONTENT_DIMS   : {N_CONTENT_DIMS}")
print(f"N_COLLAB_DIMS    : {N_COLLAB_DIMS}")

## Cell 4 — Load Data

In [ ]:
import os

def load(name, **kw):
    path = os.path.join(DATA_PATH, name)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\n  ✗ Cannot find: {path}"
            f"\n  Set DATA_PATH in Cell 3 to the folder containing your CSV files."
        )
    df = pd.read_csv(path, **kw)
    print(f"  {name:22s}  {str(df.shape):>16}  cols: {df.columns.tolist()}")
    return df

print("Loading files …")
movies       = load("movies.csv")
ratings      = load("ratings.csv")
tags         = load("tags.csv")
genome_tags  = load("genome-tags.csv")
links        = load("links.csv")
print("\n✔ All files loaded.")

## Cell 5 — Exploratory Data Analysis

In [ ]:
print("=== movies.csv ===")
display(movies.head(3))
print(movies.dtypes, "\n")

print("=== ratings.csv ===")
display(ratings.head(3))
print(f"  Total ratings  : {len(ratings):,}")
print(f"  Unique users   : {ratings['userId'].nunique():,}")
print(f"  Unique movies  : {ratings['movieId'].nunique():,}")
print(f"  Rating range   : {ratings['rating'].min()} – {ratings['rating'].max()}")
print(f"  Mean rating    : {ratings['rating'].mean():.3f}")
print()

print("=== tags.csv ===")
display(tags.head(3))
print(f"  Total tags     : {len(tags):,}")
print(f"  Unique movies  : {tags['movieId'].nunique():,}")
print()

print("=== genome-tags.csv ===")
display(genome_tags.head(3))
print(f"  Genome tags    : {len(genome_tags):,}")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Rating distribution
axes[0].hist(ratings["rating"], bins=10, color="steelblue", edgecolor="white")
axes[0].set_title("Rating Distribution"); axes[0].set_xlabel("Rating")

# Ratings per user (log scale)
rpu = ratings.groupby("userId").size()
axes[1].hist(rpu, bins=50, color="coral", edgecolor="white", log=True)
axes[1].set_title("Ratings per User (log)"); axes[1].set_xlabel("# Ratings")

# Ratings per movie (log scale)
rpm = ratings.groupby("movieId").size()
axes[2].hist(rpm, bins=50, color="mediumseagreen", edgecolor="white", log=True)
axes[2].set_title("Ratings per Movie (log)"); axes[2].set_xlabel("# Ratings")

plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=100, bbox_inches="tight")
plt.show()
print("Plot saved as eda_distributions.png")

## Cell 6 — Popularity-Based Recommender

In [ ]:
def popularity_recommender(n=10, min_votes=100):
    """
    Bayesian average:  score = (v/(v+m))*R + (m/(v+m))*C
      R = movie avg rating   v = vote count
      C = global mean        m = min_votes threshold
    Penalises movies with few ratings so they don't dominate.
    """
    stats = (ratings
             .groupby("movieId")
             .agg(vote_count=("rating","count"),
                  avg_rating =("rating","mean"))
             .reset_index())
    C = stats["avg_rating"].mean()
    stats["score"] = (
        (stats["vote_count"] / (stats["vote_count"] + min_votes)) * stats["avg_rating"] +
        (min_votes           / (stats["vote_count"] + min_votes)) * C
    )
    result = (stats[stats["vote_count"] >= min_votes]
              .sort_values("score", ascending=False)
              .head(n)
              .merge(movies[["movieId","title","genres"]], on="movieId"))
    return result[["title","genres","vote_count","avg_rating","score"]].reset_index(drop=True)

print("Top-10 Most Popular Movies (Bayesian Average):")
display(popularity_recommender(n=10))

## Cell 7 — Build Content Metadata
Merge `movies.csv` + `tags.csv` + `genome-tags.csv`.  
Each movie gets a single metadata string:  genres + all user tags + genome tag labels.


In [ ]:
# 1. Genres: replace pipe separator with space so each genre is a token
movies["genres_text"] = (movies["genres"]
                         .str.replace("|", " ", regex=False)
                         .str.replace("-", " ", regex=False)
                         .str.lower())

# 2. User tags: aggregate all tags per movie into one string
tags_agg = (tags
            .dropna(subset=["tag"])
            .assign(tag=tags["tag"].str.lower().str.strip())
            .groupby("movieId")["tag"]
            .apply(lambda x: " ".join(x))
            .reset_index()
            .rename(columns={"tag": "user_tags"}))

# 3. Genome tag labels: genome-tags.csv has tagId,tag — we just use the tag words
genome_tag_str = " ".join(genome_tags["tag"].dropna().str.lower().str.strip().unique())

# 4. Merge everything
content_df = (movies[["movieId","title","genres","genres_text"]]
              .merge(tags_agg, on="movieId", how="left"))
content_df["user_tags"] = content_df["user_tags"].fillna("")

# metadata = genres tokens + user tags (genome vocabulary enriches TF-IDF via shared tokens)
content_df["metadata"] = content_df["genres_text"] + " " + content_df["user_tags"]
content_df = content_df.reset_index(drop=True)

print(f"Content DataFrame : {content_df.shape}")
print(f"Movies with tags  : {(content_df['user_tags'] != '').sum():,}")
print(f"Movies without tags:{(content_df['user_tags'] == '').sum():,}")
display(content_df[["movieId","title","metadata"]].head(5))

## Cell 8 — TF-IDF Vectorizer + TruncatedSVD → Latent Matrix 1 (Content)

In [ ]:
print("Fitting TF-IDF vectorizer …")
tfidf = TfidfVectorizer(
    max_features=20_000,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True,       # log(1+tf) — reduces impact of very frequent terms
    min_df=2                 # ignore terms appearing in only 1 movie
)
tfidf_matrix = tfidf.fit_transform(content_df["metadata"])
print(f"  TF-IDF matrix shape : {tfidf_matrix.shape}")
print(f"  Vocabulary size     : {len(tfidf.vocabulary_):,}")

print(f"\nFitting TruncatedSVD (n_components={N_CONTENT_DIMS}) …")
svd_content     = TruncatedSVD(n_components=N_CONTENT_DIMS, random_state=42)
latent_matrix_1 = svd_content.fit_transform(tfidf_matrix)
print(f"  Latent Matrix 1 shape      : {latent_matrix_1.shape}")
print(f"  Explained variance ratio   : {svd_content.explained_variance_ratio_.sum():.3%}")

# Fast movieId → content_df row index lookup
mid_to_cidx = pd.Series(content_df.index, index=content_df["movieId"].values)
print("\n✔ Latent Matrix 1 ready.")

## Cell 9 — Content-Based Recommender

In [ ]:
def content_recommender(movie_title, n=10):
    """
    Returns n movies most similar to movie_title using cosine similarity
    on Latent Matrix 1 (TF-IDF genres+tags → TruncatedSVD).
    """
    mask = content_df["title"].str.lower().str.contains(movie_title.lower(), regex=False)
    if not mask.any():
        print(f"  ✗ No match for '{movie_title}'"); return None
    idx   = content_df[mask].index[0]
    found = content_df.loc[idx, "title"]

    sims    = cosine_similarity(latent_matrix_1[idx:idx+1], latent_matrix_1).flatten()
    top_idx = np.argsort(sims)[::-1][1:n+1]

    result = content_df.iloc[top_idx][["title","genres"]].copy()
    result["cosine_similarity"] = np.round(sims[top_idx], 4)
    print(f"[Content-Based] Seed: '{found}'")
    return result.reset_index(drop=True)

display(content_recommender("Toy Story", n=10))

## Cell 10 — Filter Ratings for Collaborative Filter

In [ ]:
print(f"Full ratings shape : {ratings.shape}")

# Keep only movies and users that meet minimum interaction thresholds
active_movies = ratings["movieId"].value_counts()
active_movies = active_movies[active_movies >= MIN_MOVIE_RATINGS].index

active_users  = ratings["userId"].value_counts()
active_users  = active_users[active_users  >= MIN_USER_RATINGS].index

rat_f = ratings[
    ratings["movieId"].isin(active_movies) &
    ratings["userId" ].isin(active_users)
].copy()

print(f"Filtered ratings   : {rat_f.shape}")
print(f"Unique users       : {rat_f['userId'].nunique():,}")
print(f"Unique movies      : {rat_f['movieId'].nunique():,}")
sparsity = 1 - len(rat_f) / (rat_f["userId"].nunique() * rat_f["movieId"].nunique())
print(f"Matrix sparsity    : {sparsity:.2%}")

## Cell 11 — User-Movie Pivot + TruncatedSVD → Latent Matrix 2 (Collaborative)

In [ ]:
print("Building User-Movie pivot table …")
umat = rat_f.pivot_table(
    index="userId", columns="movieId", values="rating", fill_value=0
)
print(f"  User-Movie matrix shape : {umat.shape}")

# Convert to sparse for efficiency
sparse_umat = csr_matrix(umat.values)

print(f"\nFitting TruncatedSVD (n_components={N_COLLAB_DIMS}) …")
svd_collab      = TruncatedSVD(n_components=N_COLLAB_DIMS, random_state=42)
latent_matrix_2 = svd_collab.fit_transform(sparse_umat)   # shape: (users, dims)
movie_latent    = svd_collab.components_.T                 # shape: (movies, dims)

print(f"  Latent Matrix 2 (users)  : {latent_matrix_2.shape}")
print(f"  Movie latent vectors     : {movie_latent.shape}")
print(f"  Explained variance ratio : {svd_collab.explained_variance_ratio_.sum():.3%}")

# Lookup helpers
cf_movie_ids = umat.columns.tolist()
mid_to_cfidx = {mid: i for i, mid in enumerate(cf_movie_ids)}
print("\n✔ Latent Matrix 2 ready.")

## Cell 12 — Collaborative Filter Recommender

In [ ]:
def collab_recommender(movie_title, n=10):
    """
    Item-item collaborative filtering via cosine similarity on Latent Matrix 2
    (movie vectors derived from User-Movie SVD decomposition).
    """
    mask = content_df["title"].str.lower().str.contains(movie_title.lower(), regex=False)
    if not mask.any():
        print(f"  ✗ No match for '{movie_title}'"); return None

    movie_id = int(content_df[mask].iloc[0]["movieId"])
    found    = content_df[mask].iloc[0]["title"]

    if movie_id not in mid_to_cfidx:
        print(f"  ✗ '{found}' has insufficient ratings for collaborative filter."); return None

    cf_i  = mid_to_cfidx[movie_id]
    sims  = cosine_similarity(movie_latent[cf_i:cf_i+1], movie_latent).flatten()
    top_i = np.argsort(sims)[::-1][1:n+1]

    rec_mids = [cf_movie_ids[i] for i in top_i]
    result   = (movies[movies["movieId"].isin(rec_mids)]
                .set_index("movieId")
                .reindex(rec_mids)
                .reset_index())
    result["cosine_similarity"] = np.round(sims[top_i], 4)
    print(f"[Collab-Filter] Seed: '{found}'")
    return result[["title","genres","cosine_similarity"]].reset_index(drop=True)

display(collab_recommender("Toy Story", n=10))

## Cell 13 — Matrix Factorization with Surprise SVD

In [ ]:
print("Building Surprise dataset …")
reader        = Reader(rating_scale=(0.5, 5.0))
data_surprise = Dataset.load_from_df(rat_f[["userId","movieId","rating"]], reader)

trainset, testset = surprise_split(data_surprise, test_size=0.2, random_state=42)
print(f"  Train size : {trainset.n_ratings:,}")
print(f"  Test  size : {len(testset):,}")

In [ ]:
print("Training Surprise SVD …")
svd_model = SVD(
    n_factors=100,    # latent factors
    n_epochs=20,      # training epochs
    lr_all=0.005,     # learning rate
    reg_all=0.02,     # regularisation
    random_state=42
)
svd_model.fit(trainset)
print("✔ Training complete.")

In [ ]:
print("Evaluating on test set …")
preds    = svd_model.test(testset)
rmse_val = accuracy.rmse(preds)
mae_val  = accuracy.mae(preds)
print(f"\nSurprise SVD  →  RMSE: {rmse_val:.4f}   MAE: {mae_val:.4f}")

In [ ]:
print("Running 5-fold cross-validation (this takes a few minutes) …")
cv = cross_validate(
    SVD(n_factors=100, n_epochs=20, random_state=42),
    data_surprise,
    measures=["RMSE","MAE"],
    cv=5,
    verbose=True
)
print(f"\nMean CV RMSE : {cv['test_rmse'].mean():.4f} ± {cv['test_rmse'].std():.4f}")
print(f"Mean CV MAE  : {cv['test_mae'].mean():.4f} ± {cv['test_mae'].std():.4f}")

In [ ]:
def surprise_recommender(user_id, n=10):
    """
    Recommend top-n unseen movies for a user via Surprise SVD predicted ratings.
    """
    rated  = set(rat_f[rat_f["userId"] == user_id]["movieId"])
    unseen = [mid for mid in cf_movie_ids if mid not in rated]

    preds = sorted(
        [(mid, svd_model.predict(user_id, mid).est) for mid in unseen],
        key=lambda x: x[1], reverse=True
    )[:n]

    rec_ids = [p[0] for p in preds]
    result  = (movies[movies["movieId"].isin(rec_ids)]
               .set_index("movieId")
               .reindex(rec_ids)
               .reset_index())
    result["predicted_rating"] = [round(p[1], 3) for p in preds]
    print(f"[Surprise SVD] Recommendations for userId={user_id}:")
    return result[["title","genres","predicted_rating"]].reset_index(drop=True)

sample_user = int(rat_f["userId"].iloc[0])
display(surprise_recommender(sample_user, n=10))

## Cell 14 — Hybrid Recommender
**Score = α × content_similarity + (1−α) × collab_similarity**

If `user_id` is supplied, final ranking is re-weighted by Surprise SVD predicted rating
(personalised hybrid = best of all three systems).


In [ ]:
def hybrid_recommender(movie_title, user_id=None, n=10, alpha=0.5):
    """
    Hybrid movie recommender combining:
      - Content similarity  (Latent Matrix 1 — TF-IDF + SVD)
      - Collab  similarity  (Latent Matrix 2 — User-Movie SVD)
      - Optional user personalisation via Surprise SVD predicted rating

    Parameters
    ----------
    movie_title : str   partial/full movie title to use as seed
    user_id     : int   optional user ID for personalisation
    n           : int   number of recommendations to return
    alpha       : float 0 = pure collab, 1 = pure content, 0.5 = balanced
    """
    # ── 1. Resolve seed movie ──────────────────────────────────────────────
    mask = content_df["title"].str.lower().str.contains(movie_title.lower(), regex=False)
    if not mask.any():
        print(f"  ✗ No match for '{movie_title}'"); return None
    seed_idx  = content_df[mask].index[0]
    seed_mid  = int(content_df.loc[seed_idx, "movieId"])
    seed_name = content_df.loc[seed_idx, "title"]

    # ── 2. Content similarities (Latent Matrix 1) ──────────────────────────
    content_sims = cosine_similarity(
        latent_matrix_1[seed_idx:seed_idx+1], latent_matrix_1
    ).flatten()

    # ── 3. Collab similarities (Latent Matrix 2) ───────────────────────────
    collab_sims = np.zeros(len(content_df))
    if seed_mid in mid_to_cfidx:
        cf_i   = mid_to_cfidx[seed_mid]
        raw_cf = cosine_similarity(movie_latent[cf_i:cf_i+1], movie_latent).flatten()
        for j, mid in enumerate(cf_movie_ids):
            rows = content_df.index[content_df["movieId"] == mid]
            if len(rows):
                collab_sims[rows[0]] = raw_cf[j]

    # ── 4. Blend ───────────────────────────────────────────────────────────
    hybrid = alpha * content_sims + (1 - alpha) * collab_sims
    hybrid[seed_idx] = -1   # exclude seed itself

    top_idx = np.argsort(hybrid)[::-1][:n * 4]
    result  = content_df.iloc[top_idx][["movieId","title","genres"]].copy()
    result["hybrid_score"] = np.round(hybrid[top_idx], 4)

    # ── 5. Optional personalisation ────────────────────────────────────────
    if user_id is not None:
        g_mean = ratings["rating"].mean()
        r_min, r_max = ratings["rating"].min(), ratings["rating"].max()

        def _pred(mid):
            try:   return svd_model.predict(user_id, mid).est
            except: return g_mean

        result["svd_pred"] = result["movieId"].apply(_pred)

        hs   = result["hybrid_score"].values
        sp   = result["svd_pred"].values
        hs_n = (hs - hs.min()) / (hs.max() - hs.min() + 1e-9)
        sp_n = (sp - r_min)    / (r_max - r_min + 1e-9)

        result["final_score"] = np.round(0.6 * hs_n + 0.4 * sp_n, 4)
        result = result.sort_values("final_score", ascending=False).head(n)
        print(f"[Hybrid Personalised]  Seed: '{seed_name}'  User: {user_id}")
        return result[["title","genres","hybrid_score","svd_pred","final_score"]].reset_index(drop=True)

    result = result.head(n)
    print(f"[Hybrid Non-Personalised]  Seed: '{seed_name}'")
    return result[["title","genres","hybrid_score"]].reset_index(drop=True)

# ── Demo ──────────────────────────────────────────────────────────────────
print("=" * 55)
display(hybrid_recommender("Toy Story", n=10))

print("\n" + "=" * 55)
display(hybrid_recommender("Toy Story", user_id=sample_user, n=10))

## Cell 15 — Evaluation Summary

In [ ]:
# ── Collab SVD reconstruction RMSE ─────────────────────────────────────────
reconstructed = np.dot(latent_matrix_2, svd_collab.components_)

u_idx_map = {u: i for i, u in enumerate(umat.index)}
m_idx_map = {m: i for i, m in enumerate(umat.columns)}

sample_eval = rat_f.sample(min(10_000, len(rat_f)), random_state=42)
y_true, y_pred_cf = [], []
for _, row in sample_eval.iterrows():
    u, m, r = row["userId"], row["movieId"], row["rating"]
    if u in u_idx_map and m in m_idx_map:
        y_true.append(r)
        y_pred_cf.append(reconstructed[u_idx_map[u], m_idx_map[m]])

rmse_cf = np.sqrt(mean_squared_error(y_true, y_pred_cf))

print("=" * 55)
print("            EVALUATION SUMMARY")
print("=" * 55)
print(f"  Collab SVD RMSE  (sklearn reconstruction) : {rmse_cf:.4f}")
print(f"  Surprise SVD RMSE (held-out test set)      : {rmse_val:.4f}")
print(f"  Surprise SVD MAE  (held-out test set)      : {mae_val:.4f}")
print("=" * 55)
print()
print("  Note: Content-based & Hybrid use cosine similarity —")
print("  RMSE is not directly applicable to those methods.")

## Cell 16 — Experiment: Different Seeds & Alpha Values

In [ ]:
test_movies = ["Matrix", "Jurassic Park", "Schindler", "Pulp Fiction", "Titanic"]
alphas      = [0.0, 0.5, 1.0]   # 0=pure collab, 0.5=balanced, 1=pure content
labels      = ["Pure Collab", "Balanced Hybrid", "Pure Content"]

for movie in test_movies:
    print("\n" + "═"*60)
    for a, label in zip(alphas, labels):
        rec = hybrid_recommender(movie, n=5, alpha=a)
        if rec is not None:
            print(f"  α={a} ({label})")
            display(rec[["title","hybrid_score"]])

## Cell 17 — Conclusion

| Method | Core Technique | Latent Structure | Strength | Weakness |
|---|---|---|---|---|
| **Popularity** | Bayesian avg rating | — | Simple, robust cold-start baseline | No personalisation |
| **Content Filter** | TF-IDF (genres+tags) + TruncatedSVD | **Latent Matrix 1** | Works for new users; semantic similarity | Ignores actual user behaviour |
| **Collaborative Filter** | User-Movie pivot + TruncatedSVD | **Latent Matrix 2** | Captures real taste patterns | Cold-start for new users/movies |
| **Surprise SVD** | Matrix Factorization (SGD) | Internal factors | Handles sparsity; calibrated RMSE | Needs tuning; no cold-start |
| **Hybrid** | Weighted blend + SVD personalisation | Both matrices | Best of all worlds | More complex |

**Data flow:**
```
movies.csv + tags.csv + genome-tags.csv
        ↓
    metadata string per movie
        ↓
    TF-IDF matrix  →  TruncatedSVD  →  Latent Matrix 1  (content sim)

ratings.csv  →  User-Movie pivot  →  TruncatedSVD  →  Latent Matrix 2  (collab sim)

ratings.csv  →  Surprise Dataset  →  SVD.fit()  →  predicted ratings (personalisation)

Latent Matrix 1 + Latent Matrix 2 + SVD predictions  →  Hybrid Score
```
